# SONARA Run 2, second pass: normalization, loudness robustness, three-class training

The first pass (`results/run2_first_pass/FINDINGS.md`) showed that ImageNet-pretrained weights and the training recipe both help, and that per-clip normalization made validation results **worse**. But every pretrained and recipe setup in that pass also had normalization switched on. This pass closes that gap and answers three questions, again **using validation data only**.

| Question | How it is tested |
| :- | :- |
| Are pretrained weights and the recipe better **without** normalization? | Setups A3, B5 and C2 repeat the first-pass leaders A2, B4 and C1 with normalization removed. |
| Does a model that sees absolute loudness **break when the recording level changes**? | Every trained model is re-scored on its validation clips with the level shifted by up to ±20 dB, at a fixed threshold. A different microphone, such as the INMP441 on the sensing nodes, will not record at the SmartEars level. |
| Does training on **three classes** (Healthy, Sick, None) reduce false alarms on healthy calls? | Setups B6 and B7 train three outputs and use the Sick probability as the distress score. Evaluation stays binary. |

Setup B8 also tests a remedy in case removing normalization helps accuracy but makes the model sensitive to level: no normalization, but a random loudness change of up to ±12 dB on every training clip.

## Rules that keep the result honest

1. **The same locked test set as the first pass.** Part 3 recomputes it from `LOCK_SEED` and stops if its id is not `bf636617`. **No cell trains on it, evaluates on it, or reads its labels.** PVSD is not loaded at all.
2. **Everything here is a validation result**, on the same three train/validation splits as the first pass, so setups from both passes can be compared seed by seed.
3. **A change counts only if it helps on most seeds.**
4. **Selection rule, fixed in advance:** the highest mean validation ROC-AUC among models that run on the Raspberry Pi 4 **and** whose distress recall and false-alarm rate both stay within 0.10 of their unshifted values when the level is shifted by ±10 dB. If two setups are not clearly different, the smaller model is chosen.
5. **The locked test set and PVSD are opened once**, in the final evaluation notebook.

**Before running:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all. The notebook asks for Google Drive access. Expect roughly 2.5 hours for 6 setups × 3 seeds; finished runs are skipped on rerun.

## 1. Setup and configuration

In [4]:
import os, re, copy, glob, hashlib, json, math, random, time
from math import gcd

import numpy as np
import pandas as pd
import soundfile as sf
from scipy.signal import resample_poly

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchvision
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau, LambdaLR

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| torch", torch.__version__, "| torchvision", torchvision.__version__)

Device: cpu | torch 2.11.0+cpu | torchvision 0.26.0+cpu


In [5]:
# ---- Data and outputs ----------------------------------------------------
REPO_URL = "https://github.com/tagalogclark123-cmyk/SONARA-GITHUB.git"
REPO_DIR = "/content/SONARA-GITHUB"
SAVE_TO_DRIVE = True          # keep results in Google Drive so a disconnect loses nothing
OUT_DIR = "/content/drive/MyDrive/SONARA/run2_second_pass" if SAVE_TO_DRIVE else "/content/run2_second_pass"
SMARTEARS_ROOT = "Smartears"
LABEL_MAP = {"Healthy": "Normal", "None": "Noise", "Sick": "Distress"}

# ---- Protocol --------------------------------------------------------------
LOCK_SEED = 2026              # fixes which 15% of SmartEars is the locked test set
EXPECTED_LOCK_ID = "bf636617" # id of the locked test set made by the first pass
EXPLORE_SEEDS = [7, 21, 42]   # each seed = one train/validation split + one training run per setup
GROUP_BY = "recording"        # clips of one source recording always stay together
TARGET_RECALL = 0.90
LEVEL_OFFSETS_DB = [-20, -10, -6, 0, 6, 10, 20]   # recording-level shifts used in the loudness check

# ---- Input (same as Run 1) -------------------------------------------------
SAMPLE_RATE = 16000
CROP_SEC = 5.0
N_FFT, HOP_LENGTH, N_MELS = 400, 160, 64

# ---- Augmentation (same as Run 1) ------------------------------------------
AUG_SHIFT_PROB, AUG_SHIFT_SEC = 0.5, 0.5
AUG_TIME_MASK_PROB, AUG_FREQ_MASK_PROB = 0.4, 0.4

BATCH_SIZE = 16
LABEL_SMOOTHING = 0.02
DROPOUT = 0.3
USE_PRETRAINED_WEIGHTS = True # False = skip the ImageNet download (the "pretrained" setups then start from random weights)
QUICK_RUN = None              # e.g. 60 = use 60 clips per class, 2 epochs, 2 seeds (code check only)

# ---- The setups to compare -------------------------------------------------
RUN1 = {"norm": "none", "pretrained": False, "optimizer": "adam", "lr": 1e-4, "weight_decay": 1e-4,
        "schedule": "plateau", "max_epochs": 50, "patience": 8, "mixup": 0.0,
        "three_class": False, "gain_prob": 0.2, "gain_db": 6.0}
NORM = {"norm": "per_clip"}
PRETRAINED = {"pretrained": True}
RECIPE = {"optimizer": "adamw", "lr": 3e-4, "weight_decay": 1e-2, "schedule": "cosine",
          "max_epochs": 40, "patience": 10, "mixup": 0.2}
THREE_CLASS = {"three_class": True}
STRONG_GAIN = {"gain_prob": 1.0, "gain_db": 12.0}


def setup(model, parent, change, *overrides):
    cfg = {"model": model, "parent": parent, "change": change, **RUN1}
    for o in overrides:
        cfg.update(o)
    return cfg


EXPERIMENTS = {
    "A3_cnn_bilstm_recipe":       setup("cnn_bilstm", "A2_cnn_bilstm_norm_recipe", "remove normalization", RECIPE),
    "B5_mobilenet_pre_recipe":    setup("mobilenetv3_small", "B4_mobilenet_norm_pre_recipe", "remove normalization", PRETRAINED, RECIPE),
    "C2_efficientnet_pre_recipe": setup("efficientnet_b0", "C1_efficientnet_norm_pre_recipe", "remove normalization", PRETRAINED, RECIPE),
    "B6_mobilenet_norm_pre_recipe_3class": setup("mobilenetv3_small", "B4_mobilenet_norm_pre_recipe", "three classes", NORM, PRETRAINED, RECIPE, THREE_CLASS),
    "B7_mobilenet_pre_recipe_3class": setup("mobilenetv3_small", "B5_mobilenet_pre_recipe", "three classes", PRETRAINED, RECIPE, THREE_CLASS),
    "B8_mobilenet_pre_recipe_gain": setup("mobilenetv3_small", "B5_mobilenet_pre_recipe", "loudness change of ±12 dB on every clip", PRETRAINED, RECIPE, STRONG_GAIN),
}

# First-pass validation results per seed: (ROC-AUC, accuracy at 0.5, false alarms at 0.90 recall).
# They are on the same locked test set and the same train/validation splits, so they pair seed by seed.
FIRST_PASS = {
    "A0_cnn_bilstm_run1":              {7: (0.9462, 0.8880, 0.1261), 21: (0.9479, 0.8781, 0.1455), 42: (0.9378, 0.8791, 0.1504)},
    "A2_cnn_bilstm_norm_recipe":       {7: (0.9435, 0.8825, 0.1546), 21: (0.9498, 0.8937, 0.1201), 42: (0.9453, 0.8802, 0.1248)},
    "B0_mobilenet_run1":               {7: (0.9349, 0.8738, 0.1832), 21: (0.9376, 0.8635, 0.1675), 42: (0.9352, 0.8835, 0.1829)},
    "B4_mobilenet_norm_pre_recipe":    {7: (0.9550, 0.9023, 0.1126), 21: (0.9636, 0.9217, 0.0660), 42: (0.9658, 0.9143, 0.0803)},
    "C1_efficientnet_norm_pre_recipe": {7: (0.9646, 0.9133, 0.0706), 21: (0.9618, 0.9195, 0.0728), 42: (0.9642, 0.9242, 0.0786)},
}
FIRST_PASS_MODEL = {"A0_cnn_bilstm_run1": "cnn_bilstm", "A2_cnn_bilstm_norm_recipe": "cnn_bilstm",
                    "B0_mobilenet_run1": "mobilenetv3_small", "B4_mobilenet_norm_pre_recipe": "mobilenetv3_small",
                    "C1_efficientnet_norm_pre_recipe": "efficientnet_b0"}

CLASS_NAMES = ["Non-distress", "Distress"]
SOURCE_INDEX = {"Normal": 0, "Distress": 1, "Noise": 2}   # Distress is index 1 in both the 2- and 3-class models
CROP_SAMPLES = int(round(CROP_SEC * SAMPLE_RATE))
N_FRAMES = CROP_SAMPLES // HOP_LENGTH
NUM_WORKERS = 2 if os.name == "posix" else 0
if QUICK_RUN:
    EXPLORE_SEEDS = EXPLORE_SEEDS[:2]


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
RUN_DIR = os.path.join(OUT_DIR, "runs")
os.makedirs(RUN_DIR, exist_ok=True)
print("Outputs:", OUT_DIR)
print(pd.DataFrame(EXPERIMENTS).T[["model", "parent", "change", "norm", "pretrained", "three_class", "gain_prob", "gain_db"]].to_string())

Mounted at /content/drive
Outputs: /content/drive/MyDrive/SONARA/run2_second_pass
                                                 model                           parent                                   change      norm pretrained three_class gain_prob gain_db
A3_cnn_bilstm_recipe                        cnn_bilstm        A2_cnn_bilstm_norm_recipe                     remove normalization      none      False       False       0.2     6.0
B5_mobilenet_pre_recipe              mobilenetv3_small     B4_mobilenet_norm_pre_recipe                     remove normalization      none       True       False       0.2     6.0
C2_efficientnet_pre_recipe             efficientnet_b0  C1_efficientnet_norm_pre_recipe                     remove normalization      none       True       False       0.2     6.0
B6_mobilenet_norm_pre_recipe_3class  mobilenetv3_small     B4_mobilenet_norm_pre_recipe                            three classes  per_clip       True        True       0.2     6.0
B7_mobilenet_pre_r

In [6]:
# Download the repository (code + audio). Takes several minutes the first time.
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
print("Repo folder:", REPO_DIR, "| exists:", os.path.isdir(REPO_DIR))

Cloning into '/content/SONARA-GITHUB'...
remote: Enumerating objects: 6382, done.
remote: Counting objects: 100% (6382/6382), done.
remote: Compressing objects: 100% (6381/6381), done.
remote: Total 6382 (delta 2), reused 6375 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (6382/6382), 2.54 GiB | 23.89 MiB/s, done.
Resolving deltas: 100% (2/2), done.
Updating files: 100% (6368/6368), done.
Repo folder: /content/SONARA-GITHUB | exists: True


## 2. Data

SmartEars only, loaded exactly as in Run 1 and the first pass: 16 kHz mono, the whole 5-second clip, binary labels (0 = Non-distress, 1 = Distress). PVSD is not loaded in this notebook.

In [7]:
SMARTEARS_NAME = re.compile(r"^(rpi\d+)-(\d{8})(\d{6})(?:-(\d+))?$")

rows = []
for folder, source in LABEL_MAP.items():
    for path in sorted(glob.glob(os.path.join(REPO_DIR, SMARTEARS_ROOT, folder, "*.wav"))):
        stem = os.path.splitext(os.path.basename(path))[0]
        m = SMARTEARS_NAME.match(stem)
        recording = f"{m.group(1)}-{m.group(2)}{m.group(3)}" if m else stem
        rows.append({"path": path, "file": f"{folder}/{os.path.basename(path)}", "original_label": folder,
                     "source": source, "label": int(source == "Distress"), "recording": recording})
manifest = pd.DataFrame(rows)
assert len(manifest) > 0, "No audio files found. Check REPO_DIR."
if QUICK_RUN:
    manifest = manifest.groupby("source").head(QUICK_RUN).reset_index(drop=True)
print(manifest.groupby(["original_label", "source", "label"]).size().to_string())


def load_audio(path):
    wav, sr = sf.read(path, dtype="float32", always_2d=True)
    wav = wav.mean(axis=1)
    if sr != SAMPLE_RATE:
        g = gcd(int(sr), SAMPLE_RATE)
        wav = resample_poly(wav, SAMPLE_RATE // g, sr // g).astype(np.float32)
    return wav


t0 = time.time()
clips = []
for i, path in enumerate(manifest["path"]):
    clips.append(np.clip(load_audio(path) * 32767.0, -32768, 32767).astype(np.int16))
    if (i + 1) % 1000 == 0:
        print(f"  loaded {i + 1}/{len(manifest)} clips ({time.time() - t0:.0f}s)")
print(f"Loaded {len(clips)} clips in {time.time() - t0:.0f}s")

original_label  source    label
Healthy         Normal    0        2000
None            Noise     0        2000
Sick            Distress  1        2000
  loaded 1000/6000 clips (12s)
  loaded 2000/6000 clips (24s)
  loaded 3000/6000 clips (36s)
  loaded 4000/6000 clips (48s)
  loaded 5000/6000 clips (59s)
  loaded 6000/6000 clips (70s)
Loaded 6000 clips in 70s


## 3. Recompute the locked test set, then split what is left

The locked test set is recomputed from `LOCK_SEED` and must have the same id as in the first pass; the cell stops otherwise. From this cell on, the code only ever indexes `dev_idx`, the other 85%.

For each exploration seed, the development data is split again into training (70% of all clips) and validation (15% of all clips). Both splits are stratified by source category and grouped by source recording.

In [8]:
def grouped_folds(indices, n_splits, seed):
    df = manifest.loc[indices]
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fold_of = np.zeros(len(df), dtype=int)
    for fold, (_, idx) in enumerate(sgkf.split(df, df["source"], df[GROUP_BY])):
        fold_of[idx] = fold
    return fold_of


all_idx = manifest.index.to_numpy()
fold_of = grouped_folds(all_idx, 20, LOCK_SEED)
locked_test_idx = all_idx[fold_of >= 17]          # 3 of 20 folds = 15%
dev_idx = all_idx[fold_of < 17]                   # the only clips this notebook uses
assert not set(manifest.loc[locked_test_idx, GROUP_BY]) & set(manifest.loc[dev_idx, GROUP_BY]), "group leakage"

lock_path = os.path.join(OUT_DIR, "locked_test_set.csv")
manifest.loc[locked_test_idx, ["file", "recording"]].to_csv(lock_path, index=False)
lock_id = hashlib.md5("\n".join(sorted(manifest.loc[locked_test_idx, "file"])).encode()).hexdigest()[:8]
print(f"Locked test set: {len(locked_test_idx)} clips, id {lock_id}, saved to {lock_path}")
print(f"Development set: {len(dev_idx)} clips")
if not QUICK_RUN:
    assert lock_id == EXPECTED_LOCK_ID, (f"Locked test set id {lock_id} differs from the first pass ({EXPECTED_LOCK_ID}). "
                                         "Stop: results would not be comparable, and first-pass test clips could leak in.")
DEV = set(dev_idx.tolist())
del locked_test_idx                                # nothing below can reach the locked clips by accident


def make_split(seed):
    fold_of = grouped_folds(dev_idx, 17, seed)
    split = {"train": dev_idx[fold_of < 14], "val": dev_idx[fold_of >= 14]}
    assert not set(manifest.loc[split["train"], GROUP_BY]) & set(manifest.loc[split["val"], GROUP_BY]), "group leakage"
    assert set(split["train"].tolist()) | set(split["val"].tolist()) <= DEV
    return split


for s in EXPLORE_SEEDS:
    sp = make_split(s)
    print(f"seed {s}: train {len(sp['train'])} {np.bincount(manifest.loc[sp['train'], 'label']).tolist()} | "
          f"val {len(sp['val'])} {np.bincount(manifest.loc[sp['val'], 'label']).tolist()}")

Locked test set: 871 clips, id bf636617, saved to /content/drive/MyDrive/SONARA/run2_second_pass/locked_test_set.csv
Development set: 5129 clips
seed 7: train 4218 [2808, 1410] | val 911 [595, 316]
seed 21: train 4235 [2812, 1423] | val 894 [591, 303]
seed 42: train 4219 [2818, 1401] | val 910 [585, 325]


## 4. Features, normalization and augmentation

The features are the same as in Run 1: mel power in dB plus delta and delta-delta, 3 × 64 × 500.

**Normalization (new).** With `norm = "per_clip"`, each channel of each clip has its own mean subtracted and is divided by its own standard deviation. A clip recorded louder or quieter then looks the same to the model. It uses only the clip itself, so no statistics are carried from one split to another.

Augmentation is applied before normalization. It is unchanged from the first pass except in setup B8, where every training clip gets a random loudness change of up to ±12 dB. Mixup, part of the recipe, blends each training clip with another one from the same batch and blends their labels in the same proportion.

In [9]:
class LogMelDeltas(nn.Module):
    """Waveform (batch, samples) -> raw dB features (batch, 3, n_mels, frames)."""

    def __init__(self):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH,
                                                        n_mels=N_MELS, center=True, power=2.0)
        self.to_db = torchaudio.transforms.AmplitudeToDB(stype="power")
        self.deltas = torchaudio.transforms.ComputeDeltas()

    def features(self, wav):
        db = self.to_db(self.mel(wav))
        d1 = self.deltas(db)
        return torch.stack([db, d1, self.deltas(d1)], dim=1)

    def forward(self, wav, lengths):
        x = self.features(wav)[..., :N_FRAMES]
        for i in torch.nonzero(lengths < CROP_SAMPLES).flatten().tolist():   # clips a few ms short of 5 s
            f = self.features(wav[i:i + 1, :max(int(lengths[i]), N_FFT)])[0]
            t = min(f.shape[-1], N_FRAMES)
            x[i, 0] = f[0].min()
            x[i, 1:] = 0.0
            x[i, :, :, :t] = f[:, :, :t]
        return x


featurizer = LogMelDeltas().to(device).eval()


def normalize(x, mode):
    if mode == "per_clip":
        mean = x.mean(dim=(2, 3), keepdim=True)
        std = x.std(dim=(2, 3), keepdim=True)
        return (x - mean) / (std + 1e-5)
    return x


def augment_features(x, cfg):
    _, _, n_freq, n_time = x.shape
    max_shift = int(AUG_SHIFT_SEC * SAMPLE_RATE / HOP_LENGTH)
    for i in range(x.size(0)):
        if max_shift and random.random() < AUG_SHIFT_PROB:
            x[i] = torch.roll(x[i], random.randint(-max_shift, max_shift), dims=2)
        ch_mean = x[i].mean(dim=(1, 2), keepdim=True)
        if random.random() < AUG_TIME_MASK_PROB:
            width = random.randint(5, n_time // 8)
            t0 = random.randint(0, n_time - width)
            x[i, :, :, t0:t0 + width] = ch_mean
        if random.random() < AUG_FREQ_MASK_PROB:
            height = random.randint(3, n_freq // 8)
            f0 = random.randint(0, n_freq - height)
            x[i, :, f0:f0 + height, :] = ch_mean
        if random.random() < cfg["gain_prob"]:
            x[i, 0] += random.uniform(-cfg["gain_db"], cfg["gain_db"])
    return x


class ClipDataset(Dataset):
    def __init__(self, indices):
        self.indices = np.asarray(indices)
        assert set(self.indices.tolist()) <= DEV, "a locked test clip was requested"
        self.labels = manifest["label"].to_numpy()
        self.sources = manifest["source"].map(SOURCE_INDEX).to_numpy()

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        seg = clips[idx][:CROP_SAMPLES].astype(np.float32) / 32768.0
        n = len(seg)
        if n < CROP_SAMPLES:
            seg = np.pad(seg, (0, CROP_SAMPLES - n))
        return torch.from_numpy(seg), int(self.labels[idx]), int(self.sources[idx]), n


def make_loader(indices, train, seed):
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(ClipDataset(indices), batch_size=BATCH_SIZE, shuffle=train, num_workers=NUM_WORKERS,
                      generator=g, pin_memory=(device.type == "cuda"))

## 5. Models

- **CNN–BiLSTM**: the Run 1 model, unchanged (1,221,186 parameters).
- **MobileNetV3-Small**: about 1.5 million parameters. With `pretrained = True` it starts from ImageNet weights and only its last layer is new.
- **EfficientNet-B0**: about 4 million parameters, ImageNet weights. Larger, but still a size that runs on a Raspberry Pi 4.

ImageNet is a photo dataset. Its weights are used only as a starting point; every layer is then trained on SmartEars. The three feature channels (dB, delta, delta-delta) take the place of the three colour channels.

In the three-class setups the last layer has three outputs (Healthy, Sick, None) and the Sick probability is the distress score.

In [10]:
class CNN_BiLSTM(nn.Module):
    def __init__(self, n_mels=64, num_classes=2, cnn_channels=64, lstm_hidden=128, dropout=DROPOUT):
        super().__init__()
        self.conv1 = nn.Conv2d(3, cnn_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(cnn_channels)
        self.conv2 = nn.Conv2d(cnn_channels, cnn_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(cnn_channels)
        self.pool = nn.MaxPool2d(kernel_size=(2, 2))
        self.lstm = nn.LSTM(input_size=cnn_channels * (n_mels // 4), hidden_size=lstm_hidden,
                            num_layers=1, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(lstm_hidden * 2, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        b, c, fq, t = x.shape
        x = x.permute(0, 3, 1, 2).reshape(b, t, c * fq)
        _, (h_n, _) = self.lstm(x)
        return self.fc(self.dropout(torch.cat([h_n[-2], h_n[-1]], dim=1)))


def build_model(name, pretrained, num_classes=2):
    use_weights = "IMAGENET1K_V1" if (pretrained and USE_PRETRAINED_WEIGHTS) else None
    if name == "cnn_bilstm":
        return CNN_BiLSTM(num_classes=num_classes)
    if name == "mobilenetv3_small":
        m = torchvision.models.mobilenet_v3_small(weights=use_weights)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes)
        return m
    if name == "efficientnet_b0":
        m = torchvision.models.efficientnet_b0(weights=use_weights)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes)
        return m
    raise ValueError(name)


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def time_ms(fn, runs=20):
    for _ in range(3):
        fn()
    ts = []
    for _ in range(runs):
        t = time.perf_counter()
        fn()
        ts.append((time.perf_counter() - t) * 1000)
    return round(float(np.median(ts)), 1)


# Size and a CPU timing proxy (this Colab CPU, NOT the Raspberry Pi 4) for the deployability check.
x_cpu = torch.randn(1, 3, N_MELS, N_FRAMES)
deploy = []
with torch.no_grad():
    for name in sorted({cfg["model"] for cfg in EXPERIMENTS.values()}):
        mdl = build_model(name, pretrained=False).eval()
        deploy.append({"model": name, "parameters": count_params(mdl), "size_MB": round(count_params(mdl) * 4 / 1e6, 2),
                       "cpu_ms_per_clip": time_ms(lambda: mdl(x_cpu))})
deploy = pd.DataFrame(deploy)
deploy.to_csv(os.path.join(OUT_DIR, "model_size_and_cpu_time.csv"), index=False)
print(f"CPU threads: {torch.get_num_threads()} (Colab proxy, not the Pi)")
print(deploy.to_string(index=False))

CPU threads: 1 (Colab proxy, not the Pi)
            model  parameters  size_MB  cpu_ms_per_clip
       cnn_bilstm     1221186     4.88             70.8
  efficientnet_b0     4010110    16.04             43.8
mobilenetv3_small     1519906     6.08             11.7


## 6. Training and the loudness check

Training is the same as in the first pass: weighted cross-entropy with label smoothing of 0.02, and the best checkpoint is the epoch with the lowest validation loss. Every setup in this pass uses the recipe (AdamW, learning rate 3e-4 with warm-up and cosine decay, mixup, at most 40 epochs).

**Loudness check.** After training, the best checkpoint scores its validation clips again with the level shifted by each value in `LEVEL_OFFSETS_DB`. A shift of +10 dB is what the same sound would look like through a microphone or gain setting about three times more sensitive. The threshold is the one chosen at the original level and is **not** re-tuned, because a deployed gateway cannot re-tune it. A robust model keeps roughly the same recall and false-alarm rate at every shift.

In [11]:
def forward_features(wav, n, cfg, training=False, level_offset_db=0.0):
    with torch.no_grad():
        x = featurizer(wav, n)
        if level_offset_db:
            x[:, 0] += level_offset_db                # a level change is a constant offset on the dB channel
        if training:
            x = augment_features(x, cfg)
        return normalize(x, cfg["norm"])


def run_epoch(model, loader, cfg, criterion, optimizer=None, level_offset_db=0.0):
    training = optimizer is not None
    model.train(training)
    total_loss, total, probs, labels = 0.0, 0, [], []
    with torch.set_grad_enabled(training):
        for wav, y, src, n in loader:
            wav, y, src = wav.to(device), y.to(device), src.to(device)
            target = src if cfg["three_class"] else y
            x = forward_features(wav, n, cfg, training, level_offset_db)
            if training and cfg["mixup"] > 0:
                lam = float(np.random.beta(cfg["mixup"], cfg["mixup"]))
                perm = torch.randperm(x.size(0), device=device)
                logits = model(lam * x + (1 - lam) * x[perm])
                loss = lam * criterion(logits, target) + (1 - lam) * criterion(logits, target[perm])
            else:
                logits = model(x)
                loss = criterion(logits, target)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            else:
                probs.append(F.softmax(logits, dim=1)[:, 1].cpu().numpy())   # index 1 = Distress
                labels.append(y.cpu().numpy())
            total_loss += loss.item() * y.size(0)
            total += y.size(0)
    if training:
        return total_loss / total, None, None
    return total_loss / total, np.concatenate(probs), np.concatenate(labels)


def train_setup(name, seed, split):
    cfg = EXPERIMENTS[name]
    max_epochs = 2 if QUICK_RUN else cfg["max_epochs"]
    n_classes = 3 if cfg["three_class"] else 2
    set_seed(seed)
    model = build_model(cfg["model"], cfg["pretrained"], n_classes).to(device)
    targets = (manifest.loc[split["train"], "source"].map(SOURCE_INDEX) if cfg["three_class"]
               else manifest.loc[split["train"], "label"])
    counts = np.bincount(targets, minlength=n_classes)
    weights = torch.tensor(counts.mean() / counts, dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=LABEL_SMOOTHING)
    if cfg["optimizer"] == "adamw":
        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    if cfg["schedule"] == "cosine":
        warm = 2
        scheduler = LambdaLR(optimizer, lambda e: (e + 1) / warm if e < warm else
                             max(0.02, 0.5 * (1 + math.cos(math.pi * (e - warm) / max(1, max_epochs - warm)))))
    else:
        scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2, min_lr=1e-6)
    train_loader = make_loader(split["train"], True, seed)
    val_loader = make_loader(split["val"], False, seed)

    history, best = [], {"loss": float("inf")}
    stale = 0
    for epoch in range(1, max_epochs + 1):
        train_loss, _, _ = run_epoch(model, train_loader, cfg, criterion, optimizer)
        val_loss, p_val, y_val = run_epoch(model, val_loader, cfg, criterion)
        scheduler.step(val_loss) if cfg["schedule"] == "plateau" else scheduler.step()
        improved = val_loss < best["loss"] - 1e-4
        if improved:
            best, stale = {"loss": val_loss, "epoch": epoch, "p_val": p_val, "y_val": y_val,
                           "state": copy.deepcopy(model.state_dict())}, 0
        else:
            stale += 1
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                        "val_auc": roc_auc_score(y_val, p_val), "best": improved})
        print(f"  [{name} seed {seed}] epoch {epoch:02d} train {train_loss:.4f} val {val_loss:.4f} "
              f"auc {history[-1]['val_auc']:.4f}{' *' if improved else ''}")
        if stale >= cfg["patience"]:
            break
    model.load_state_dict(best.pop("state"))
    return model.eval(), best, pd.DataFrame(history), val_loader, criterion


def level_check(model, cfg, val_loader, criterion, threshold):
    """Validation ROC-AUC, recall and false-alarm rate at each level shift, threshold held fixed."""
    out = {}
    for off in LEVEL_OFFSETS_DB:
        _, p, y = run_epoch(model, val_loader, cfg, criterion, level_offset_db=float(off))
        flagged = (p >= threshold).astype(int)
        out[str(off)] = {"auc": roc_auc_score(y, p), "recall": float(flagged[y == 1].mean()),
                         "false_alarms": float(flagged[y == 0].mean())}
    return out


def choose_threshold(y, p, target=TARGET_RECALL):
    ok = [t for t in np.round(np.arange(0.01, 0.995, 0.005), 3)
          if recall_score(y, (p >= t).astype(int), zero_division=0) >= target]
    return float(max(ok)) if ok else 0.5


def validation_metrics(y, p, sources):
    pred = (p >= 0.5).astype(int)
    thr = choose_threshold(y, p)
    flagged = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, flagged, labels=[0, 1]).ravel()
    return {"val_roc_auc": roc_auc_score(y, p), "val_pr_auc": average_precision_score(y, p),
            "val_accuracy_at_0.5": accuracy_score(y, pred),
            "val_distress_f1_at_0.5": f1_score(y, pred, zero_division=0),
            "val_distress_recall_at_0.5": recall_score(y, pred, zero_division=0),
            "threshold_for_recall_0.90": thr,
            "val_false_alarm_rate_at_recall_0.90": fp / max(fp + tn, 1),
            "val_false_alarms_normal": float(flagged[sources == "Normal"].mean()),
            "val_false_alarms_noise": float(flagged[sources == "Noise"].mean())}

## 7. Run every setup on every seed

Each finished run writes a small JSON file, its validation predictions and its training history to `OUT_DIR/runs`. Finished runs are skipped when this cell is run again, so after a disconnect just run the notebook again.

In [12]:
for seed in EXPLORE_SEEDS:
    split = make_split(seed)
    for name, cfg in EXPERIMENTS.items():
        base = os.path.join(RUN_DIR, f"{name}_seed{seed}")
        if os.path.exists(base + ".json"):
            print(f"skip (done): {name}_seed{seed}")
            continue
        t0 = time.time()
        model, best, history, val_loader, criterion = train_setup(name, seed, split)
        sources = manifest.loc[split["val"], "source"].to_numpy()
        metrics = validation_metrics(best["y_val"], best["p_val"], sources)
        levels = level_check(model, cfg, val_loader, criterion, metrics["threshold_for_recall_0.90"])
        result = {"setup": name, "seed": seed, "lock_id": lock_id,
                  **{k: v for k, v in cfg.items()}, "parameters": count_params(model),
                  "epochs_run": len(history), "best_epoch": best["epoch"], "best_val_loss": best["loss"],
                  "train_minutes": (time.time() - t0) / 60, **metrics, "level_check": levels}
        np.savez(base + "_val_predictions.npz", p_val=best["p_val"], y_val=best["y_val"], rows_val=split["val"])
        history.to_csv(base + "_history.csv", index=False)
        with open(base + ".json", "w") as f:        # written last: marks the run as complete
            json.dump(result, f, indent=1, default=float)
        print(f"DONE {name}_seed{seed}: val AUC {result['val_roc_auc']:.4f} | acc@0.5 {result['val_accuracy_at_0.5']:.4f} | "
              f"false alarms at 0.90 recall {result['val_false_alarm_rate_at_recall_0.90']:.4f} | "
              f"recall at -10/0/+10 dB {levels['-10']['recall']:.2f}/{levels['0']['recall']:.2f}/{levels['10']['recall']:.2f} | "
              f"{result['train_minutes']:.1f} min\n")
        del model

skip (done): A3_cnn_bilstm_recipe_seed7
skip (done): B5_mobilenet_pre_recipe_seed7
skip (done): C2_efficientnet_pre_recipe_seed7
skip (done): B6_mobilenet_norm_pre_recipe_3class_seed7
skip (done): B7_mobilenet_pre_recipe_3class_seed7
skip (done): B8_mobilenet_pre_recipe_gain_seed7
skip (done): A3_cnn_bilstm_recipe_seed21
skip (done): B5_mobilenet_pre_recipe_seed21
skip (done): C2_efficientnet_pre_recipe_seed21
skip (done): B6_mobilenet_norm_pre_recipe_3class_seed21
skip (done): B7_mobilenet_pre_recipe_3class_seed21
skip (done): B8_mobilenet_pre_recipe_gain_seed21
skip (done): A3_cnn_bilstm_recipe_seed42
skip (done): B5_mobilenet_pre_recipe_seed42
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 153MB/s]


  [C2_efficientnet_pre_recipe seed 42] epoch 01 train 0.5380 val 0.4153 auc 0.9267 *
  [C2_efficientnet_pre_recipe seed 42] epoch 02 train 0.4759 val 0.3721 auc 0.9363 *
  [C2_efficientnet_pre_recipe seed 42] epoch 03 train 0.4335 val 0.3268 auc 0.9504 *
  [C2_efficientnet_pre_recipe seed 42] epoch 04 train 0.4078 val 0.3141 auc 0.9508 *
  [C2_efficientnet_pre_recipe seed 42] epoch 05 train 0.3884 val 0.2992 auc 0.9595 *
  [C2_efficientnet_pre_recipe seed 42] epoch 06 train 0.3706 val 0.2804 auc 0.9618 *
  [C2_efficientnet_pre_recipe seed 42] epoch 07 train 0.3580 val 0.2845 auc 0.9626
  [C2_efficientnet_pre_recipe seed 42] epoch 08 train 0.3474 val 0.2951 auc 0.9699
  [C2_efficientnet_pre_recipe seed 42] epoch 09 train 0.3456 val 0.2833 auc 0.9682
  [C2_efficientnet_pre_recipe seed 42] epoch 10 train 0.3256 val 0.2666 auc 0.9681 *
  [C2_efficientnet_pre_recipe seed 42] epoch 11 train 0.3235 val 0.2718 auc 0.9684
  [C2_efficientnet_pre_recipe seed 42] epoch 12 train 0.2883 val 0.2747 a

100%|██████████| 9.83M/9.83M [00:00<00:00, 124MB/s]


  [B6_mobilenet_norm_pre_recipe_3class seed 42] epoch 01 train 0.9119 val 0.7679 auc 0.9025 *
  [B6_mobilenet_norm_pre_recipe_3class seed 42] epoch 02 train 0.7908 val 0.6302 auc 0.9180 *
  [B6_mobilenet_norm_pre_recipe_3class seed 42] epoch 03 train 0.7411 val 0.6125 auc 0.9251 *
  [B6_mobilenet_norm_pre_recipe_3class seed 42] epoch 04 train 0.6925 val 0.5392 auc 0.9401 *
  [B6_mobilenet_norm_pre_recipe_3class seed 42] epoch 05 train 0.6602 val 0.5291 auc 0.9425 *
  [B6_mobilenet_norm_pre_recipe_3class seed 42] epoch 06 train 0.6305 val 0.5164 auc 0.9479 *
  [B6_mobilenet_norm_pre_recipe_3class seed 42] epoch 07 train 0.6081 val 0.5148 auc 0.9500 *
  [B6_mobilenet_norm_pre_recipe_3class seed 42] epoch 08 train 0.5868 val 0.5023 auc 0.9477 *
  [B6_mobilenet_norm_pre_recipe_3class seed 42] epoch 09 train 0.5661 val 0.5095 auc 0.9563
  [B6_mobilenet_norm_pre_recipe_3class seed 42] epoch 10 train 0.5489 val 0.4682 auc 0.9502 *
  [B6_mobilenet_norm_pre_recipe_3class seed 42] epoch 11 train

## 8. Validation results

All numbers in this part are **validation** results, mean ± standard deviation over the exploration seeds. Rows marked "first pass" are copied from the first-pass run for reference.

- **ROC-AUC** is the selection metric. It does not depend on a threshold.
- **False alarms at 0.90 recall** is the share of Non-distress clips flagged when the threshold is set to catch 90% of Distress clips. Lower is better.

In [13]:
def pm(s, digits=3):
    s = pd.Series(s, dtype=float)
    return f"{s.mean():.{digits}f} ± {s.std(ddof=1):.{digits}f}" if len(s) > 1 else f"{s.mean():.{digits}f}"


runs = pd.DataFrame([json.load(open(p)) for p in sorted(glob.glob(os.path.join(RUN_DIR, "*.json")))])
runs = runs[runs["setup"].isin(EXPERIMENTS) & runs["seed"].isin(EXPLORE_SEEDS) & (runs["lock_id"] == lock_id)]
runs.drop(columns="level_check").to_csv(os.path.join(OUT_DIR, "all_runs.csv"), index=False)

summary = []
for name, per_seed_values in FIRST_PASS.items():
    v = [per_seed_values[s] for s in EXPLORE_SEEDS if s in per_seed_values]
    summary.append({"Setup": name, "Pass": "first pass", "Seeds": len(v), "ROC-AUC": pm([x[0] for x in v]),
                    "Accuracy @0.5": pm([x[1] for x in v]), "False alarms @0.90 recall": pm([x[2] for x in v])})
for name, cfg in EXPERIMENTS.items():
    g = runs[runs["setup"] == name]
    if len(g):
        summary.append({"Setup": name, "Pass": "second pass", "Seeds": len(g), "ROC-AUC": pm(g["val_roc_auc"]),
                        "Accuracy @0.5": pm(g["val_accuracy_at_0.5"]),
                        "False alarms @0.90 recall": pm(g["val_false_alarm_rate_at_recall_0.90"]),
                        "Distress F1 @0.5": pm(g["val_distress_f1_at_0.5"]),
                        "…on Normal": pm(g["val_false_alarms_normal"]), "…on Noise": pm(g["val_false_alarms_noise"]),
                        "Best epoch": pm(g["best_epoch"], 1)})
summary = pd.DataFrame(summary).fillna("")
summary.to_csv(os.path.join(OUT_DIR, "second_pass_validation_summary.csv"), index=False)
pd.set_option("display.width", 250)
print("VALIDATION results (not test results)")
print(summary.to_string(index=False))

VALIDATION results (not test results)
                              Setup        Pass  Seeds       ROC-AUC Accuracy @0.5 False alarms @0.90 recall Distress F1 @0.5    …on Normal     …on Noise  Best epoch
                 A0_cnn_bilstm_run1  first pass      3 0.944 ± 0.005 0.882 ± 0.005             0.141 ± 0.013                                                         
          A2_cnn_bilstm_norm_recipe  first pass      3 0.946 ± 0.003 0.885 ± 0.007             0.133 ± 0.019                                                         
                  B0_mobilenet_run1  first pass      3 0.936 ± 0.001 0.874 ± 0.010             0.178 ± 0.009                                                         
       B4_mobilenet_norm_pre_recipe  first pass      3 0.961 ± 0.006 0.913 ± 0.010             0.086 ± 0.024                                                         
    C1_efficientnet_norm_pre_recipe  first pass      3 0.964 ± 0.002 0.919 ± 0.005             0.074 ± 0.004                        

In [14]:
# Effect of each change: a setup compared with its parent, seed by seed (same seed = same split).
def per_seed(name):
    """{seed: (ROC-AUC, false alarms at 0.90 recall)} from this pass or from the first pass."""
    if name in FIRST_PASS:
        return {s: (v[0], v[2]) for s, v in FIRST_PASS[name].items()}
    g = runs[runs["setup"] == name]
    return {int(r["seed"]): (r["val_roc_auc"], r["val_false_alarm_rate_at_recall_0.90"]) for _, r in g.iterrows()}


comparisons = []
for name, cfg in EXPERIMENTS.items():
    a, b = per_seed(name), per_seed(cfg["parent"])
    common = sorted(set(a) & set(b))
    if not common:
        continue
    d_auc = np.array([a[s][0] - b[s][0] for s in common])
    d_fa = np.array([a[s][1] - b[s][1] for s in common])
    comparisons.append({"Setup": name, "Compared with": cfg["parent"], "Change": cfg["change"], "Seeds": len(common),
                        "ROC-AUC change": round(d_auc.mean(), 4), "Seeds better (AUC)": int((d_auc > 0).sum()),
                        "False-alarm change": round(d_fa.mean(), 4), "Seeds better (false alarms)": int((d_fa < 0).sum())})
comparisons = pd.DataFrame(comparisons)
comparisons.to_csv(os.path.join(OUT_DIR, "second_pass_comparisons.csv"), index=False)
print(comparisons.to_string(index=False))
print("\nA positive ROC-AUC change and a negative false-alarm change are improvements. Count a change as "
      f"helping only if it is better on most of the {len(EXPLORE_SEEDS)} seeds; otherwise report no clear effect.")

                              Setup                   Compared with                                  Change  Seeds  ROC-AUC change  Seeds better (AUC)  False-alarm change  Seeds better (false alarms)
               A3_cnn_bilstm_recipe       A2_cnn_bilstm_norm_recipe                    remove normalization      3          0.0064                   3             -0.0179                            1
            B5_mobilenet_pre_recipe    B4_mobilenet_norm_pre_recipe                    remove normalization      3          0.0027                   3             -0.0135                            3
         C2_efficientnet_pre_recipe C1_efficientnet_norm_pre_recipe                    remove normalization      3          0.0064                   3             -0.0131                            2
B6_mobilenet_norm_pre_recipe_3class    B4_mobilenet_norm_pre_recipe                           three classes      3          0.0008                   1              0.0001                            1


## 9. Loudness check

Each row is one setup, averaged over the seeds. The threshold was set at the original level (0 dB) to catch 90% of Distress clips and then held fixed.

- **Recall** should stay near 0.90 at every shift. If it collapses when the level drops, a quieter microphone would miss distress.
- **False alarms** should stay near their 0 dB value. If they jump when the level rises, a louder microphone would flag healthy birds.

Setups with per-clip normalization cannot see the level at all, so their rows are flat by construction. First-pass setups without normalization (A0, B0) were not checked.

In [15]:
level_rows = []
for name in EXPERIMENTS:
    g = runs[runs["setup"] == name]
    if len(g) == 0:
        continue
    row = {"Setup": name, "Normalization": EXPERIMENTS[name]["norm"]}
    for metric, label in [("recall", "Recall"), ("false_alarms", "False alarms"), ("auc", "ROC-AUC")]:
        for off in LEVEL_OFFSETS_DB:
            row[f"{label} {off:+d} dB"] = round(float(np.mean([lc[str(off)][metric] for lc in g["level_check"]])), 3)
    row["Largest recall change at ±10 dB"] = round(max(abs(row[f"Recall {off:+d} dB"] - row["Recall +0 dB"]) for off in (-10, 10)), 3)
    row["Largest false-alarm change at ±10 dB"] = round(max(abs(row[f"False alarms {off:+d} dB"] - row["False alarms +0 dB"]) for off in (-10, 10)), 3)
    level_rows.append(row)
level_table = pd.DataFrame(level_rows)
level_table.to_csv(os.path.join(OUT_DIR, "second_pass_level_check.csv"), index=False)
for label in ["Recall", "False alarms", "ROC-AUC"]:
    print(f"{label} at each level shift (threshold fixed at its 0 dB value):")
    print(level_table[["Setup"] + [f"{label} {off:+d} dB" for off in LEVEL_OFFSETS_DB]].to_string(index=False), "\n")
print(level_table[["Setup", "Normalization", "Largest recall change at ±10 dB", "Largest false-alarm change at ±10 dB"]].to_string(index=False))

Recall at each level shift (threshold fixed at its 0 dB value):
                              Setup  Recall -20 dB  Recall -10 dB  Recall -6 dB  Recall +0 dB  Recall +6 dB  Recall +10 dB  Recall +20 dB
               A3_cnn_bilstm_recipe          0.388          0.801         0.869         0.901         0.901          0.863          0.616
            B5_mobilenet_pre_recipe          0.720          0.867         0.888         0.901         0.892          0.882          0.841
         C2_efficientnet_pre_recipe          0.655          0.863         0.883         0.901         0.901          0.893          0.793
B6_mobilenet_norm_pre_recipe_3class          0.901          0.901         0.901         0.901         0.901          0.901          0.901
     B7_mobilenet_pre_recipe_3class          0.763          0.881         0.896         0.901         0.895          0.876          0.821
       B8_mobilenet_pre_recipe_gain          0.850          0.894         0.897         0.901         0.901 

## 10. Ranking by the selection rule

The rule was fixed before this run: the highest mean validation ROC-AUC among models that run on the Raspberry Pi 4 and whose recall and false-alarm rate both stay within 0.10 of their unshifted values at ±10 dB. First-pass setups with normalization pass the loudness condition by construction; first-pass setups without normalization were not checked and are marked as such.

In [16]:
rank_rows = []
for name, per in FIRST_PASS.items():
    v = [per[s] for s in EXPLORE_SEEDS if s in per]
    normalized = "norm" in name
    rank_rows.append({"setup": name, "pass": "first", "model": FIRST_PASS_MODEL[name], "seeds": len(v),
                      "val_roc_auc": np.mean([x[0] for x in v]), "false_alarms_at_090": np.mean([x[2] for x in v]),
                      "recall_change_at_10dB": 0.0 if normalized else np.nan,
                      "false_alarm_change_at_10dB": 0.0 if normalized else np.nan,
                      "level_robust": "yes (normalized)" if normalized else "not checked"})
for name in EXPERIMENTS:
    g = runs[runs["setup"] == name]
    if len(g) == 0:
        continue
    lt = level_table[level_table["Setup"] == name].iloc[0]
    change, fa_change = float(lt["Largest recall change at ±10 dB"]), float(lt["Largest false-alarm change at ±10 dB"])
    rank_rows.append({"setup": name, "pass": "second", "model": EXPERIMENTS[name]["model"], "seeds": len(g),
                      "val_roc_auc": g["val_roc_auc"].mean(),
                      "false_alarms_at_090": g["val_false_alarm_rate_at_recall_0.90"].mean(),
                      "recall_change_at_10dB": change, "false_alarm_change_at_10dB": fa_change,
                      "level_robust": "yes" if max(change, fa_change) <= 0.10 else "NO"})
ranking = (pd.DataFrame(rank_rows).merge(deploy[["model", "parameters", "cpu_ms_per_clip"]], on="model")
           .sort_values("val_roc_auc", ascending=False).round(4))
ranking.to_csv(os.path.join(OUT_DIR, "second_pass_ranking.csv"), index=False)
print(ranking.to_string(index=False))

incomplete = ranking[(ranking["pass"] == "second") & (ranking["seeds"] < len(EXPLORE_SEEDS))]["setup"].tolist()
if incomplete:
    print("\nNot all seeds finished for:", incomplete, "- rerun the notebook before reading the ranking.")
eligible = ranking[ranking["level_robust"].str.startswith("yes")]
print(f"\nLeading setup that passes the loudness condition: {eligible.iloc[0]['setup']} "
      f"(validation ROC-AUC {eligible.iloc[0]['val_roc_auc']:.4f})")
print("Before choosing it, check in Part 8 whether it is clearly ahead of the next smaller model on most seeds.")
print("These are validation results. The locked test set (id " + lock_id + ") and PVSD have not been used.")

                              setup   pass             model  seeds  val_roc_auc  false_alarms_at_090  recall_change_at_10dB  false_alarm_change_at_10dB     level_robust  parameters  cpu_ms_per_clip
         C2_efficientnet_pre_recipe second   efficientnet_b0      3       0.9699               0.0609                  0.038                       0.013              yes     4010110             43.8
       B8_mobilenet_pre_recipe_gain second mobilenetv3_small      3       0.9698               0.0722                  0.007                       0.006              yes     1519906             11.7
     B7_mobilenet_pre_recipe_3class second mobilenetv3_small      3       0.9685               0.0638                  0.025                       0.020              yes     1519906             11.7
            B5_mobilenet_pre_recipe second mobilenetv3_small      3       0.9642               0.0728                  0.034                       0.021              yes     1519906             11.7
    C

## 11. What to do with these results

1. Copy the four `second_pass_*.csv` files from `OUT_DIR` into the repo under `results/`, together with this executed notebook.
2. Record which changes helped on most seeds and which did not, including the loudness check.
3. Choose the final setup by the rule in Part 10. The final evaluation notebook then retrains that setup and the Run 1 CNN–BiLSTM on 5 seeds and opens the locked test set and PVSD once.
4. Do not report these validation numbers as the system's performance.